In [1]:
import os
import re
from pathlib import Path

import numpy as np
import torch
from natsort import natsorted
from transformers import PeAudioVideoModel, PeAudioVideoProcessor

# Update these paths
MODEL_DIR = Path("/home/amin/hf_models/manual/facebook--pe-av-large")
DATA_DIR = Path("/home/amin/Research/Representation/Movie/data/Data/Separated")
OUTPUT_PATH = Path("/home/amin/Research/Representation/Movie/outputs/7T_MOVIE1_CC1_v2/pe-av-large/peav_large_embeds_2s.pt")

BATCH_SIZE = 1
MAX_RUNS = 18
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
LOCAL_FILES_ONLY = True

print(f"device is {DEVICE}")

device is cuda


In [2]:
os.environ["HF_HUB_DOWNLOAD_TIMEOUT"] = "300"

if LOCAL_FILES_ONLY:
    model = PeAudioVideoModel.from_pretrained(MODEL_DIR, local_files_only=True).to(DEVICE)
    processor = PeAudioVideoProcessor.from_pretrained(MODEL_DIR, local_files_only=True)
else:
    model = PeAudioVideoModel.from_pretrained(MODEL_DIR).to(DEVICE)
    processor = PeAudioVideoProcessor.from_pretrained(MODEL_DIR)

print("Model & Processor Ready")


Loading weights:   0%|          | 0/1035 [00:00<?, ?it/s]

Model & Processor Ready


In [3]:
def extract_part_number(filename: Path):
    match = re.search(r"part[_-]?(\d+)", filename.stem, re.IGNORECASE)
    if match:
        return int(match.group(1))
    return None

# Gather all video/audio chunk pairs across all runs
all_videos = []
all_audios = []

for run_idx in range(1, MAX_RUNS + 1):
    vid_chunk_dir = DATA_DIR / f"Video{run_idx}" / f"Video{run_idx}_chunks"
    aud_chunk_dir = DATA_DIR / f"Audio{run_idx}" / f"Audio{run_idx}_chunks"

    if not vid_chunk_dir.exists() or not aud_chunk_dir.exists():
        print(f"Warning: Missing dir for run {run_idx}")
        continue

    vids = natsorted(vid_chunk_dir.glob("*.mp4"))
    auds = natsorted(aud_chunk_dir.glob("*.wav"))
    if len(vids) == 0 or len(auds) == 0:
        print(f"Warning: No chunks found for run {run_idx}")
        continue

    vid_parts = {}
    for v in vids:
        part_num = extract_part_number(v)
        if part_num is not None:
            vid_parts[part_num] = v

    aud_parts = {}
    for a in auds:
        part_num = extract_part_number(a)
        if part_num is not None:
            aud_parts[part_num] = a

    common_parts = sorted(set(vid_parts.keys()) & set(aud_parts.keys()))

    for part_num in common_parts:
        all_videos.append(vid_parts[part_num])
        all_audios.append(aud_parts[part_num])

    print(f"Run {run_idx}: {len(common_parts)} paired chunks (video: {len(vids)}, audio: {len(auds)})")

paired = list(zip(all_videos, all_audios))
print(f"\n{'=' * 60}")
print(f"Total: {len(paired)} paired 2s segments")
print(f"{'=' * 60}\n")

if len(paired) == 0:
    raise RuntimeError("No paired segments found.")


Run 1: 106 paired chunks (video: 106, audio: 106)
Run 2: 111 paired chunks (video: 111, audio: 111)
Run 3: 94 paired chunks (video: 94, audio: 94)
Run 4: 32 paired chunks (video: 32, audio: 32)
Run 5: 42 paired chunks (video: 42, audio: 42)
Run 7: 114 paired chunks (video: 114, audio: 114)
Run 8: 130 paired chunks (video: 130, audio: 130)
Run 9: 125 paired chunks (video: 125, audio: 125)
Run 10: 42 paired chunks (video: 42, audio: 42)
Run 11: 91 paired chunks (video: 91, audio: 91)
Run 12: 102 paired chunks (video: 102, audio: 102)
Run 13: 72 paired chunks (video: 72, audio: 72)
Run 14: 42 paired chunks (video: 42, audio: 42)
Run 15: 117 paired chunks (video: 117, audio: 117)
Run 16: 115 paired chunks (video: 115, audio: 115)
Run 17: 128 paired chunks (video: 128, audio: 128)
Run 18: 40 paired chunks (video: 40, audio: 40)

Total: 1503 paired 2s segments



In [4]:
# Extract embeddings
video_out, audio_out, av_out = [], [], []
clip_names = []
run_ids = []

with torch.inference_mode():
    for i in range(0, len(paired), BATCH_SIZE):
        batch_pairs = paired[i : i + BATCH_SIZE]
        v_batch = [str(v) for v, _ in batch_pairs]
        a_batch = [str(a) for _, a in batch_pairs]

        inputs = processor(videos=v_batch, audio=a_batch, return_tensors="pt").to(DEVICE)

        if DEVICE.type == "cuda":
            with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
                outputs = model(**inputs)
        else:
            outputs = model(**inputs)

        video_out.append(outputs.video_embeds.float().cpu())
        audio_out.append(outputs.audio_embeds.float().cpu())
        av_out.append(outputs.audio_video_embeds.float().cpu())

        for v, _ in batch_pairs:
            clip_names.append(v.stem)
            run_ids.append(v.parent.parent.name)

        if (i // BATCH_SIZE) % 50 == 0 or i == 0:
            print(f"Processed {i + len(batch_pairs)}/{len(paired)}")

video_embeds = torch.cat(video_out, dim=0)
audio_embeds = torch.cat(audio_out, dim=0)
av_embeds = torch.cat(av_out, dim=0)

print("\nDone!")
print(f"Video embeddings: {video_embeds.shape}")
print(f"Audio embeddings: {audio_embeds.shape}")
print(f"AV embeddings:    {av_embeds.shape}")


/home/amin/miniconda3/envs/peav/lib/python3.10/site-packages/transformers/video_processing_utils.py:873: UserWarning: `torchcodec` is not installed and cannot be used to decode the video by default. Falling back to `torchvision`. Note that `torchvision` decoding is deprecated and will be removed in future versions. 
  warnings.warn(
/home/amin/miniconda3/envs/peav/lib/python3.10/site-packages/transformers/video_utils.py:536: UserWarning: Using `torchvision` for video decoding is deprecated and will be removed in future versions. Please use `torchcodec` instead.
  warnings.warn(
/home/amin/miniconda3/envs/peav/lib/python3.10/site-packages/torchvision/io/_video_deprecation_warning.py:9: UserWarning: The video decoding and encoding capabilities of torchvision are deprecated from version 0.22 and will be removed in version 0.24. We recommend that you migrate to TorchCodec, where we'll consolidate the future decoding/encoding capabilities of PyTorch: https://github.com/pytorch/torchcodec
  

OutOfMemoryError: CUDA out of memory. Tried to allocate 68.00 MiB. GPU 0 has a total capacity of 11.50 GiB of which 47.62 MiB is free. Including non-PyTorch memory, this process has 11.43 GiB memory in use. Of the allocated memory 10.56 GiB is allocated by PyTorch, and 688.79 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

torch.save(
    {
        "video": video_embeds,
        "audio": audio_embeds,
        "av": av_embeds,
        "clip_names": clip_names,
        "run_ids": run_ids,
    },
    OUTPUT_PATH,
)

print(f"saved to {OUTPUT_PATH}")


In [ ]:
npz_path = OUTPUT_PATH.with_suffix(".npz")
np.savez(
    npz_path,
    video=video_embeds.numpy(),
    audio=audio_embeds.numpy(),
    av=av_embeds.numpy(),
    clip_names=np.array(clip_names),
    run_ids=np.array(run_ids),
)
print(f"also saved to {npz_path}")
